# 4. Arm two: the external evaluation

Reproduces the section *RQ3: external evaluation on arm two*. Arm two is real commit history. It is an
**external evaluation**.

The labels of both arms are recomputed here from the per-execution medians and schedule positions
(`data/redmi9t_executions.csv`, `data/arm2_executions.csv`). The training set is rebuilt from them and the models are
retrained from it at fixed settings. Which pairs enter arm two is read from `data/arm2_population.json`: building that
population needs the mined repositories, which are not part of the release. Everything after it (the labels, the
decisive set, the one test, the robustness criteria, the rule comparisons) is recomputed here and checked.

In [1]:
import warnings
from common import *
from scipy.stats import binomtest
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
A4 = load(RES / "arm2_evaluation.json")
POP = load(DATA / "arm2_population.json")
REVIEW = load(RES / "posthoc_sensitivity_checks.json")
REVIEW2 = load(RES / "posthoc_followup_checks.json")

## The labels, from the executions

Each group's session drift is removed by regressing the log of each execution median on its schedule position, with
one intercept per role. The label of a pair is the Cliff's δ of the second revision's adjusted values over the first's:
slower above 0.33, faster below −0.33, indistinguishable in between. On arm one this rebuilds the training set: the
variant-to-variant contrasts that move at least one of the eleven candidate features and are not indistinguishable.

In [2]:
CANDIDATES_11 = ["d_treeNonConstWidgetCount", "d_treeMaxWidgetNestingDepth", "d_treeListRenderingStrategy",
                 "d_rootBuildReturnsConstWidget", "d_treeConstWidgetCount", "d_helperReferenceCount",
                 "d_treeCyclomaticComplexity", "d_treeIterationCount", "d_iterationWidgetCount",
                 "d_valueObjectAllocCount", "d_helperWidgetCount"]

# Arm one: the training set.
_, ADJ1 = drift_adjusted(load_executions("redmi9t_executions.csv"))
P1 = pd.read_csv(DATA / "redmi9t_pairs_labelled.csv", dtype={"group": str})
P1["delta"] = [cliffs_delta(ADJ1[(g, r)], ADJ1[(g, l)]) for g, l, r in zip(P1.group, P1.left, P1.right)]
moves = ~(P1[CANDIDATES_11] == 0).all(axis=1)
rebuilt = P1[(P1.contrast_type == "mut_mut") & moves & (P1.delta.abs() > CUT)].reset_index(drop=True)
train = pd.read_csv(DATA / "mut_mut_training_set.csv", dtype={"group": str})
check("training pairs rebuilt", len(rebuilt), 819)
check("same pairs, same order", (rebuilt.pair_id.to_numpy() == train.pair_id.to_numpy()).all(), True)
check("largest difference to the stored delta", np.abs(rebuilt.delta.to_numpy() - train.delta.to_numpy()).max(), 0, 1e-12)
check("labels identical", ((rebuilt.delta > CUT).astype(int).to_numpy() == train.label.to_numpy()).all(), True)

# Arm two: every labelled pair of the population.
BETA2, ADJ2 = drift_adjusted(load_executions("arm2_executions.csv"))
allp = {p["pair_key"]: p for p in POP["population_pairs"] + POP["p6_sensitivity_pairs"]}
stored = {r["pair_key"]: r for r in A4["labels"]["per_pair"]}
lab = {k: {"delta_P": cliffs_delta(ADJ2[(allp[k]["group"], allp[k]["b"])], ADJ2[(allp[k]["group"], allp[k]["a"])])}
       for k in stored}
check("arm-two pairs labelled", len(lab), 182)
check("largest difference to the evaluated delta", max(abs(lab[k]["delta_P"] - stored[k]["delta_P"]) for k in lab), 0, 1e-12)
check("largest difference to the evaluated drift slopes",
      max(abs(BETA2[g] - b) for g, b in A4["labels"]["beta_by_group"].items()), 0, 1e-12)

## The models

Training data: the 819 decisive variant-to-variant contrasts of arm one, each used twice, as the pair and as its
reverse (features negated, label flipped), so that every model is symmetric by construction. Nothing is tuned. The
primary model is a random forest (500 trees, `max_features="sqrt"`, other defaults). The other three learners are fitted
the same way, on the same eight features, and reported beside the forest without a test. The eight features are those of the eleven candidates that move in at least 5% of
arm-two pairs and in at least 3 scopes, a rule that uses no label.

In [3]:
DROPPED = {"d_rootBuildReturnsConstWidget", "d_treeIterationCount", "d_treeListRenderingStrategy"}
cols8 = [f for f in CANDIDATES_11 if f not in DROPPED]
check("training pairs", len(train), 819)
check("training pairs: second version slower", train.label.sum(), 564)

def build(learner):
    if learner == "random_forest":
        return RandomForestClassifier(n_estimators=500, max_features="sqrt", min_samples_leaf=1, max_depth=None,
                                      bootstrap=True, class_weight=None, random_state=SEED)
    if learner == "logreg_balanced":
        return Pipeline([("variance", VarianceThreshold()), ("scale", RobustScaler()),
                         ("model", LogisticRegression(class_weight="balanced", random_state=SEED))])
    if learner == "rbf_svm":
        return Pipeline([("variance", VarianceThreshold()), ("scale", RobustScaler()), ("model", SVC(random_state=SEED))])
    if learner == "lightgbm":
        return Pipeline([("variance", VarianceThreshold()),
                         ("model", LGBMClassifier(deterministic=True, n_jobs=1, random_state=SEED, verbosity=-1))])

def fit_pairs(learner, X, y):
    """Each pair is used twice, as given and reversed (features negated, label flipped)."""
    return build(learner).fit(np.vstack([X, -X]), np.concatenate([y, 1 - y]))

def fit_symmetric(learner, cols):
    return fit_pairs(learner, train[cols].to_numpy(float), train.label.to_numpy(int))

MODELS8 = {m: fit_symmetric(m, cols8) for m in ("random_forest", "lightgbm", "rbf_svm", "logreg_balanced")}

def antisymmetric(model, X, svm=False):
    """s(D) = f(D) - f(-D): the prediction for a pair and for its reverse always disagree."""
    X = np.asarray(X, float)
    if svm:
        return 0.5 * (model.decision_function(X) - model.decision_function(-X))
    return model.predict_proba(X)[:, 1] - model.predict_proba(-X)[:, 1]

def correct(y, s):
    """1 where the predicted direction matches the measured direction of the second revision."""
    return (np.sign(s).astype(int) == (2 * np.asarray(y, int) - 1)).astype(float)

## From the upper bound to the test set

Pairs of revisions of the same scope. Three filters are applied in order: the render check, the re-screen, and the
exclusion of *indistinguishable* pairs (position-adjusted |Cliff's δ| within 0.33 of zero).

In [4]:
pairs = {p["pair_key"]: p for p in POP["population_pairs"]}
deferred = {p["pair_key"]: p for p in POP["p6_sensitivity_pairs"]}   # touched by the deferred criterion
fp = A4["final_population"]
removed_render = {r["pair_key"] for r in fp["removals"] if r["step"] == "P4"}
final = set(fp["final_pair_keys"])
decisive = sorted(k for k in final if k not in deferred and abs(lab[k]["delta_P"]) > CUT)
indist = sorted(k for k in final if k not in deferred and abs(lab[k]["delta_P"]) <= CUT)

def size(keys):
    keys = list(keys)
    return len(keys), len({allp[k]["scope"] for k in keys}), len({allp[k]["repo"] for k in keys})

funnel = pd.DataFrame({
    "upper bound": size(allp),
    "of which under the population rule": size(pairs),
    "of which touched by the deferred criterion": size(deferred),
    "after the render check": size(set(allp) - removed_render),
    "after the re-screen": size(final),
    "of which not touched by the deferred criterion": size(final - set(deferred)),
    "indistinguishable, of those": size(indist),
    "decisive: the test set": size(decisive),
}, index=["pairs", "scopes", "repositories"]).T
print(funnel.to_string())
for row, want in {"upper bound": (296, 40, 22), "of which under the population rule": (263, 36, 19),
                  "of which touched by the deferred criterion": (33, 4, 3), "after the render check": (215, 28, 17),
                  "after the re-screen": (182, 28, 17), "of which not touched by the deferred criterion": (158, 26, 15),
                  "decisive: the test set": (110, 18, 13)}.items():
    for col, w in zip(["pairs", "scopes", "repositories"], want):
        check(f"{row}: {col}", funnel.loc[row, col], w)
check("indistinguishable pairs", len(indist), 48)
check("indistinguishable share", len(indist) / 158, 0.304, 5e-4)

keys = decisive
y = np.array([lab[k]["delta_P"] > 0 for k in keys]).astype(int)     # 1: the second revision is slower
repo = np.array([allp[k]["repo"] for k in keys]); scope = np.array([allp[k]["scope"] for k in keys])
ts = np.array([allp[k]["orientation"] == "timestamp" for k in keys])
def X(cols): return np.array([[allp[k]["delta12"][c] for c in cols] for k in keys], float)
check("pairs ordered by commit timestamp", ts.sum(), 107)
check("largest repository's decisive pairs", pd.Series(repo).value_counts().max(), 35)
check("second revision slower", y.sum(), 92); check("second revision faster", (1 - y).sum(), 18)

                                                pairs  scopes  repositories
upper bound                                       296      40            22
of which under the population rule                263      36            19
of which touched by the deferred criterion         33       4             3
after the render check                            215      28            17
after the re-screen                               182      28            17
of which not touched by the deferred criterion    158      26            15
indistinguishable, of those                        48      15             9
decisive: the test set                            110      18            13


## The one test

One random forest (500 trees, fixed settings), trained once on the arm-one variant-to-variant stratum with eight features.
Test: one-sided exact binomial test of directional agreement against 0.5 on the decisive pairs. Two robustness criteria:
(B) a two-stage cluster bootstrap (repositories, then scopes) whose one-sided 95% lower bound must exceed 0.5, and
(L) leave-one-repository-out agreement must stay above 0.5.

In [5]:
def cluster_bootstrap_lower(c, repo, scope, draws=2000, alpha=0.05):
    repos = sorted(set(repo)); scopes_of = {r: sorted(set(scope[repo == r])) for r in repos}
    idx = {s: np.flatnonzero(scope == s) for s in sorted(set(scope))}
    rng = np.random.default_rng(SEED); stats = np.empty(draws)
    for d in range(draws):
        a = t = 0.0
        for ri in rng.integers(0, len(repos), len(repos)):
            sc = scopes_of[repos[ri]]
            for si in rng.integers(0, len(sc), len(sc)):
                a += c[idx[sc[si]]].sum(); t += len(idx[sc[si]])
        stats[d] = a / t
    return float(np.quantile(stats, alpha))

s_rf = antisymmetric(MODELS8["random_forest"], X(cols8))
c = correct(y, s_rf); k, n = int(c.sum()), len(y)
p_one = binomtest(k, n, 0.5, alternative="greater").pvalue
lo, hi = clopper_pearson(k, n)
pt = A4["primary_test"]
check("agreement, k", k, 86); check("agreement, n", n, 110)
check("agreement", k / n, pt["agreement"], 1e-12); check("agreement (reported)", k / n, 0.782, 5e-4)
check("one-sided binomial p", p_one, pt["p_upper"], 1e-15)
check("Clopper-Pearson low", lo, 0.693, 5e-4); check("Clopper-Pearson high", hi, 0.855, 5e-4)
B = cluster_bootstrap_lower(c, repo, scope)
check("(B) lower bound", B, A4["robustness_B"]["lower_bound"], 1e-12); check("(B) lower bound (reported)", B, 0.706, 5e-4)
L = {r: (k - c[repo == r].sum()) / (n - (repo == r).sum()) for r in sorted(set(repo))}
check("(L) lowest leave-one-repository-out agreement", min(L.values()), 0.769, 5e-4)
check("(L) highest", max(L.values()), 0.800, 5e-4)
check("(L) repositories", len(L), 13)
check("outcome row", A4["outcome"]["row"], 2)
print(f"{k}/{n} = {k/n:.3f}, CI [{lo:.3f}, {hi:.3f}], p = {p_one:.2g}; (B) {B:.3f}; (L) {min(L.values()):.3f}-{max(L.values()):.3f}")

86/110 = 0.782, CI [0.693, 0.855], p = 1.1e-09; (B) 0.706; (L) 0.769-0.800


### The reference a fair coin is not

92 of the 110 labels point the same way, so agreement with 0.5 is the weakest reference available. A guesser that
ignores the code but predicts "second slower" as often as the model does (0.818) agrees in expectation on
q·p + (1−q)(1−p), where q = 92/110. This comparison is post-hoc and carries no test.

In [6]:
q, pr = y.mean(), (s_rf > 0).mean()
check("model predicts 'second slower'", pr, 0.818, 5e-4)
check("expected agreement of the guesser", q * pr + (1 - q) * (1 - pr), 0.714, 5e-4)
check("agreement if the 48 indistinguishable pairs count as wrong", k / (n + len(indist)), 0.544, 5e-4)
check("model right where the second revision is faster, k", int(c[y == 0].sum()), 7)

## Post-hoc comparison with a one-feature rule

Computed after the result was seen. No p-value, and no outcome row changes. The rule: the second revision is slower iff
the non-constant widget count rises (zero change predicts "not slower"). Beside it, the constant rule that always predicts
"second revision slower" is scored. Both are post-hoc and carry no test.

In [7]:
COUNT = "d_treeNonConstWidgetCount"
cnt = X([COUNT])[:, 0]
rule = (cnt > 0).astype(int)
rc = (rule == y).astype(float)
# Always predict "second revision slower": it ignores the code and shows the label imbalance.
const = np.ones(len(y), dtype=int)
check("always 'second slower': agreement k", (const == y).sum(), 92)
check("always 'second slower' (reported)", (const == y).mean(), 0.836, 5e-4)
check("always 'second slower' beats the forest by", (const == y).sum() - k, 6)
check("count rule agreement, k", rc.sum(), 97); check("count rule (reported)", rc.mean(), 0.882, 5e-4)
check("pairs where the count moves", (cnt != 0).sum(), 94)
check("  count rule right there", rc[cnt != 0].sum(), 88)
check("count rises", (cnt > 0).sum(), 89); check("count falls", (cnt < 0).sum(), 5)
check("forest predicts 'slower' when the count rises", (s_rf[cnt > 0] > 0).sum(), 78)
check("forest predicts 'slower' when the count falls", (s_rf[cnt < 0] > 0).sum(), 0)
check("pairs where forest and count rule disagree: count rule right", int(((rc == 1) & (c == 0)).sum()), 17)
check("pairs where forest and count rule disagree: forest right", int(((rc == 0) & (c == 1)).sum()), 6)
check("pairs where the two disagree", int((rc != c).sum()), 23)

# Per pair, a 2,000-draw bootstrap of the adjusted delta (executions of each side resampled independently).
import zlib
lo_, hi_ = [], []
for k_ in keys:
    Lv, Rv = ADJ2[(allp[k_]["group"], allp[k_]["a"])], ADJ2[(allp[k_]["group"], allp[k_]["b"])]
    g_ = np.random.default_rng([SEED, zlib.crc32(k_.encode())])
    iL, iR = g_.integers(0, len(Lv), (2000, len(Lv))), g_.integers(0, len(Rv), (2000, len(Rv)))
    boot = np.array([cliffs_delta(Rv[iR[t]], Lv[iL[t]]) for t in range(2000)])
    lo_.append(np.percentile(boot, 2.5)); hi_.append(np.percentile(boot, 97.5))
clear = (np.array(lo_) > 0) | (np.array(hi_) < 0)
check("decisive pairs whose interval excludes zero", clear.sum(), 99)
check("  forest agrees", int(c[clear].sum()), 79)
check("  count rule agrees", int(rc[clear].sum()), 93)
one = REVIEW["C_single_feature_rule_arm2"]["one_pair_per_scope"]
check("one pair per scope: forest", one["model_agree_k"], 14); check("one pair per scope: count rule", one["count_rule_agree_k"], 15)
check("one pair per scope: scopes", one["n_scopes"], 18)

### The same comparison on the training stratum, and further rules

On the 819 training pairs, the forest and the count rule are scored out-of-fold: five group-disjoint folds at seed
20260803, the forest refitted inside each training fold with every pair also reversed. The permutation importance is
the fold-size-weighted drop in out-of-fold agreement, ten permutations per feature per fold. It orders the features
for thirteen further hand-readable rules, written from arm one and scored on both arms after the result was seen:
one sign rule per other feature, tie-breakers on the count (`lex`), sign votes (`vote`) and two shallow trees. On arm
two each rule is also scored with every pair read in reverse (220 readings). Post-hoc, no test.

In [8]:
X1, y1, g1 = train[cols8].to_numpy(float), train.label.to_numpy(int), train.group.to_numpy()
FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(len(y1)), y1, g1))
rng_ = np.random.default_rng(SEED)
s_oof = np.full(len(y1), np.nan); drops = {f: [] for f in cols8}; sizes = []
for tr, te in FOLDS:
    m_ = fit_pairs("random_forest", X1[tr], y1[tr])
    s_ = antisymmetric(m_, X1[te]); s_oof[te] = s_; base_ = correct(y1[te], s_).mean(); sizes.append(len(te))
    for j, f in enumerate(cols8):
        d_ = []
        for _ in range(10):
            Xp = X1[te].copy(); Xp[:, j] = rng_.permutation(Xp[:, j])
            d_.append(base_ - correct(y1[te], antisymmetric(m_, Xp)).mean())
        drops[f].append(float(np.mean(d_)))
importance = {f: float(np.dot(np.array(sizes) / sum(sizes), drops[f])) for f in cols8}
order = sorted(importance, key=lambda f: -importance[f])
dc1 = train[COUNT].to_numpy()
dev_forest, dev_rule = correct(y1, s_oof).mean(), ((dc1 > 0) == (y1 == 1)).mean()
check("arm one, forest out-of-fold (seed 20260803)", dev_forest, 0.753, 5e-4)
check("arm one, forest out-of-fold, k", correct(y1, s_oof).sum(), 617)
check("arm one, count rule", dev_rule, 0.611, 5e-4)
check("arm one, count rises", (dc1 > 0).sum(), 377); check("arm one, count falls", (dc1 < 0).sum(), 359)
check("arm one, forest predicts 'slower' where the count falls", int((s_oof[dc1 < 0] > 0).sum()), 154)
check("most important feature", order[0] == COUNT, True)

def rule_direction(x, y):
    return 1 if ((x > 0) == (y == 1)).sum() >= ((x < 0) == (y == 1)).sum() else -1

def fit_rule(name, X, y):
    """A rule fitted on (X, y); returns predict(Z) -> True where the second revision is slower."""
    idx = {f: j for j, f in enumerate(cols8)}
    dirs = {f: rule_direction(X[:, idx[f]], y) for f in cols8}
    if name.startswith("sign_"):
        f = name[5:]
        return lambda Z: dirs[f] * Z[:, idx[f]] > 0
    if name.startswith("lex"):
        feats = [COUNT] + [f for f in order if f != COUNT][: int(name[3:]) - 1]
        def lex(Z):
            out, open_ = np.zeros(len(Z), bool), np.ones(len(Z), bool)
            for f in feats:
                v = dirs[f] * Z[:, idx[f]]
                out[open_ & (v > 0)] = True
                open_ &= v == 0
            return out
        return lex
    if name.startswith("vote"):
        feats = order[: int(name[4:])]
        return lambda Z: sum(dirs[f] * np.sign(Z[:, idx[f]]) for f in feats) > 0
    tree = DecisionTreeClassifier(max_depth=int(name[6:]), random_state=SEED).fit(X, y)
    return lambda Z: tree.predict(Z) == 1

X2 = X(cols8)
rules = {}
for name in [f"sign_{f}" for f in order] + ["lex2", "lex3", "vote2", "vote3", "tree_d2", "tree_d3"]:
    oof = np.zeros(len(y1), bool)
    for tr, te in FOLDS:
        oof[te] = fit_rule(name, X1[tr], y1[tr])(X1[te])
    p_ = fit_rule(name, X1, y1)
    a2, a2r = p_(X2), p_(-X2)
    rules[name] = {"arm one, out-of-fold": int((oof == (y1 == 1)).sum()), "arm two": int((a2 == (y == 1)).sum()),
                   "arm two, both readings (of 220)": int((a2 == (y == 1)).sum() + (a2r == (y == 0)).sum()),
                   "arm two, predicts 'slower'": int(a2.sum())}
rules = pd.DataFrame(rules).T
print(rules.to_string())
further = rules.drop(index=f"sign_{COUNT}")
check("further rules", len(further), 13)
check("best further rule on arm two does not beat the count rule", further["arm two"].max() <= rc.sum(), True)
check("tie-breakers reproduce the count rule's 97", (rules.loc[["lex2", "lex3"], "arm two"] == 97).all(), True)
check("largest gain of a tie-breaker, both readings", (rules.loc[["lex2", "lex3"], "arm two, both readings (of 220)"]
                                                      - rules.loc[f"sign_{COUNT}", "arm two, both readings (of 220)"]).max(), 4)
check("depth-three tree is best on arm one", rules["arm one, out-of-fold"].idxmax() == "tree_d3", True)
check("depth-three tree, arm one", rules.loc["tree_d3", "arm one, out-of-fold"] / len(y1), 0.763, 5e-4)
check("depth-three tree, arm two", rules.loc["tree_d3", "arm two"], 95)
check("depth-three tree predicts 'slower'", rules.loc["tree_d3", "arm two, predicts 'slower'"], 107)
check("depth-three tree, both readings", rules.loc["tree_d3", "arm two, both readings (of 220)"], 140)
check("forest, both readings", 2 * k, 172)

                                  arm one, out-of-fold  arm two  arm two, both readings (of 220)  arm two, predicts 'slower'
sign_d_treeNonConstWidgetCount                     500       97                              192                          89
sign_d_helperWidgetCount                           379       18                              111                           0
sign_d_treeCyclomaticComplexity                    341       18                               64                           2
sign_d_treeConstWidgetCount                        376       23                               68                          13
sign_d_helperReferenceCount                        390       19                              112                           1
sign_d_treeMaxWidgetNestingDepth                   417       26                              114                          12
sign_d_valueObjectAllocCount                       352       60                              150                          54


## The other three learners (descriptive, none tested)

LightGBM, the RBF support vector machine and balanced logistic regression, fitted on the same eight features, are
scored through the same path. Agreement is compared with the frozen forest by discordant pairs, never by a test.

In [9]:
desc = {}
for m_, svm in (("lightgbm", False), ("rbf_svm", True), ("logreg_balanced", False)):
    s = antisymmetric(MODELS8[m_], X(cols8), svm)
    cc = correct(y, s)
    desc[m_] = (int(cc.sum()), int(((cc == 1) & (c == 0)).sum()), int(((cc == 0) & (c == 1)).sum()), float((s > 0).mean()))
rows = pd.DataFrame(desc, index=["agree (of 110)", "row right / forest wrong", "forest right / row wrong", "predicts 'second slower'"]).T
print(rows.to_string())
want = {"lightgbm": (90, 6, 2), "rbf_svm": (49, 4, 41), "logreg_balanced": (45, 9, 50)}
for m_, w in want.items():
    for j, lab_ in enumerate(["agreement k", "row right / forest wrong", "forest right / row wrong"]):
        check(f"{m_}: {lab_}", rows.loc[m_].iloc[j], w[j])
check("logistic regression predicts 'second slower' on", desc["logreg_balanced"][3], 0.373, 5e-4)
for m_, ci in (("lightgbm", (0.733, 0.885)), ("rbf_svm", (0.351, 0.543)), ("logreg_balanced", (0.316, 0.507))):
    a, b = clopper_pearson(desc[m_][0], 110)
    check(f"{m_} CI low", a, ci[0], 5e-4); check(f"{m_} CI high", b, ci[1], 5e-4)



                 agree (of 110)  row right / forest wrong  forest right / row wrong  predicts 'second slower'
lightgbm                   90.0                       6.0                       2.0                  0.854545
rbf_svm                    49.0                       4.0                      41.0                  0.463636
logreg_balanced            45.0                       9.0                      50.0                  0.372727


## Further description of the frozen forest (none tested)

Both orientations of every pair are scored for the AUC and average precision; the Matthews coefficient is on one copy of
each pair. Subgroup agreements are frozen counts.

In [10]:
check("ROC AUC, both orientations", pt["roc_auc_both_orientations"], 0.846, 5e-4)
check("average precision, both orientations", pt["average_precision_both_orientations"], 0.820, 5e-4)
check("Matthews coefficient, one copy", pt["mcc_one_copy"], 0.237, 5e-4)
check("scope-weighted agreement", A4["descriptions"]["scope_weighted_agreement"], 0.761, 5e-4)
sg = A4["subgroups_descriptive"]
for name, key, kk, nn in [("adjacent revisions", "adjacent", 18, 23), ("non-adjacent revisions", "non_adjacent", 65, 84),
                          ("signatures never seen in training", "novel_signature", 53, 64),
                          ("resumed sessions excluded", "resumed_session_excluded", 66, 87)]:
    check(f"{name}: k", sg[key]["k"], kk); check(f"{name}: n", sg[key]["n"], nn)
for t, (kk, nn) in {"0": (27, 31), "1": (21, 34), "2": (38, 45)}.items():
    check(f"schedule-position tercile {t}: k", sg["schedule_position_tercile"][t]["k"], kk)
    check(f"schedule-position tercile {t}: n", sg["schedule_position_tercile"][t]["n"], nn)
check("decisive pairs from a generated (not authored) dependency file", sg["not_authored_fixture"]["n"], 1)

# Why the linear learners fall below one half: arm-two deltas mostly rise with the count (frozen coefficients).
coef = REVIEW2["q1_4_logreg"]["coef_per_robust_scaled_unit"]
print({k_.replace("d_", ""): round(v, 2) for k_, v in coef.items()})
check("logistic coefficient, non-constant widget count", coef["d_treeNonConstWidgetCount"], 1.36, 5e-3)
check("logistic coefficient, constant widget count", coef["d_treeConstWidgetCount"], -0.26, 5e-3)

{'helperReferenceCount': 0.25, 'helperWidgetCount': 0.25, 'iterationWidgetCount': 0.01, 'treeConstWidgetCount': -0.26, 'treeCyclomaticComplexity': -0.11, 'treeMaxWidgetNestingDepth': -0.16, 'treeNonConstWidgetCount': 1.36, 'valueObjectAllocCount': -0.21}


## Figures

The figures for this notebook's sections, written to `figures/` as PNG (300 dpi) and SVG. They plot only values checked above. The images carry no title or note; the caption in the report explains them.

In [11]:
plt = figure_style()

# The frozen forest against the count rule: arm two (a), and arm one's training stratum (b).
rows = [("frozen random forest (the one test)", k / n, (lo, hi), DARK, None),
        ("count rule (post-hoc)", rc.mean(), None, MID, None),
        ("label imbalance: share second-slower", (const == y).mean(), None, LIGHT, None),
        ("rate-matched code-blind guesser", q * pr + (1 - q) * (1 - pr), None, "white", "///")]
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1.3, 1]})
for i, (label, v, ci, colour, hatch) in enumerate(rows):
    a.barh(i, v, color=colour, edgecolor=DARK, hatch=hatch)
    if ci:
        a.errorbar(v, i, xerr=[[v - ci[0]], [ci[1] - v]], color="black", capsize=3, lw=1)
    a.text((ci[1] if ci else v) + 0.01, i, f"{v:.3f}", va="center", fontsize=7.5)
a.set_yticks(range(len(rows)), [r[0] for r in rows], fontsize=7.5)
a.invert_yaxis()
a.axvline(0.5, color=DARK, ls=":", lw=1)
a.set_xlim(0, 1.05)
a.set_xlabel(f"directional agreement, {n} decisive pairs")
a.set_title("(a)", loc="left")
for i, (label, v, colour) in enumerate([("random forest", dev_forest, DARK), ("count rule", dev_rule, MID)]):
    b.barh(i, v, color=colour, edgecolor=DARK)
    b.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=7.5)
b.set_yticks(range(2), ["random forest", "count rule"], fontsize=7.5)
b.invert_yaxis()
b.axvline(0.5, color=DARK, ls=":", lw=1)
b.set_xlim(0, 1.05)
b.set_xlabel(f"agreement, {len(y1)} decisive training pairs (out-of-fold)")
b.set_title("(b)", loc="left")
fig.tight_layout()
save_figure(fig, "arm2_agreement")

wrote figures/arm2_agreement.png and .svg


## The forest for `spm screen`

The screening tool in the `spm` package embeds this forest. Every tree is written as flat arrays (feature index,
threshold, left and right child, and the leaf's share of *slower*; a node is a leaf when its left child is −1), with
the 819 training and 110 decisive vectors and the score this notebook gives each, so the tool's own tests can check
that it reproduces every score.

In [12]:
def tree_arrays(t):
    p = t.value[:, 0, 1] / t.value[:, 0, :].sum(axis=1)
    leaf = t.children_left == -1
    return {"feature": [int(f) if not lf else -1 for f, lf in zip(t.feature, leaf)],
            "threshold": [float(x) if not lf else 0.0 for x, lf in zip(t.threshold, leaf)],
            "left": t.children_left.astype(int).tolist(), "right": t.children_right.astype(int).tolist(),
            "pSlower": [float(x) if lf else 0.0 for x, lf in zip(p, leaf)]}

forest = MODELS8["random_forest"]
features = [f.removeprefix("d_") for f in cols8]
model = {"name": "forest_s_support_v1", "features": features,
         "inference": "f(D) = mean leaf pSlower, left iff D[f] <= threshold; s = f(D) - f(-D); slower iff s > 0",
         "trees": [tree_arrays(e.tree_) for e in forest.estimators_]}
parity = {"features": features}
for name, Xs, ys in (("arm1_mut_mut", X1, y1), ("arm2_decisive", X2, y)):
    parity[name] = {"x": Xs.astype(int).tolist(), "s": [float(v) for v in antisymmetric(forest, Xs)], "y": [int(v) for v in ys]}
check("exported scores reproduce the forest's 86 of 110", int(((np.array(parity["arm2_decisive"]["s"]) > 0) == (y == 1)).sum()), 86)
EXPORT = ANALYSIS / "spm_export"
EXPORT.mkdir(exist_ok=True)
(EXPORT / "forest_s_support_v1.json").write_text(json.dumps(model, separators=(",", ":")))
(EXPORT / "forest_parity_vectors.json").write_text(json.dumps(parity, separators=(",", ":")))
print("wrote spm_export/forest_s_support_v1.json and forest_parity_vectors.json")

wrote spm_export/forest_s_support_v1.json and forest_parity_vectors.json


In [13]:
report()

128 / 128 reported figures reproduced


,claim,value,reported,ok
0,training pairs rebuilt,819,819,True
1,"same pairs, same order",True,True,True
2,largest difference to the stored delta,0.0,0,True
3,labels identical,True,True,True
4,arm-two pairs labelled,182,182,True
...,...,...,...,...
123,schedule-position tercile 2: n,45,45,True
124,decisive pairs from a generated (not authored)...,1,1,True
125,"logistic coefficient, non-constant widget count",1.361855,1.36,True
126,"logistic coefficient, constant widget count",-0.262499,-0.26,True
